In [3]:
import pandas as pd


In [5]:
data = pd.read_excel("Club2024.xlsx", sheet_name=None)
print(data.keys())

dict_keys(['Races', 'Overview', 'Cobbler', 'Mamores', 'Traprain', 'Allermuir', 'Arrochar', 'Blisco', 'Moffat', 'Caerketton', 'Tap O North', 'Perris Horseshoe', 'Caerketton Down', 'Run O Mill', 'Manor Water', 'Skyloop', 'Tinto'])


In [6]:
results = data[2:]

KeyError: slice(2, None, None)

In [7]:
metadata = data.pop("Races", "Overview")


In [12]:
results = data.pop("Overview")

In [13]:
data.keys()

dict_keys(['Cobbler', 'Mamores', 'Traprain', 'Allermuir', 'Arrochar', 'Blisco', 'Moffat', 'Caerketton', 'Tap O North', 'Perris Horseshoe', 'Caerketton Down', 'Run O Mill', 'Manor Water', 'Skyloop', 'Tinto'])

In [14]:
results

,Posn,Name,Ran,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,Total


In [15]:
len(data)

15

In [16]:
NON_SHR = ["Blisco", "Perris Horseshoe"]
#Race data that doesn't fit the SHR table format (Usually the two british champs races


In [19]:
mamores = data["Mamores"]

In [28]:
mamores = mamores[mamores["Club"].str.startswith("Carnethy")]
#mamores['Timedelta'] = pd.to_timedelta(mamores['Time'])
mamores['Cat Pos'] = mamores.groupby('Cat')['Time'].rank(method='min', ascending=True).astype(int)

mamores = mamores.sort_values(by=['Cat', 'Time'])

mamores

,Pos,Runner,Club,Cat,Time,Cat Pos
18,19,Naomi Lang,Carnethy HRC,F,00:55:31,1
101,102,Helen Fallas,Carnethy HRC,F,01:16:28,2
2,3,Kieran Cooper,Carnethy HRC,M,00:48:51,1
11,12,Felix Wilson,Carnethy HRC,M,00:52:46,2
12,13,Csoban Balogh,Carnethy HRC,M,00:52:57,3
15,16,Ifan Oldfield,Carnethy HRC,M,00:54:35,4
27,28,Iain Gilmore,Carnethy HRC,M,00:56:56,5
20,21,Andrew Fallas,Carnethy HRC,M40,00:56:06,1
68,69,Michael Reid,Carnethy HRC,M40,01:05:35,2
77,78,Christopher Busby,Carnethy HRC,M40,01:10:50,3


In [55]:
# Function to add 'Cat Pos' for each dataframe
def add_cat_pos(df):
    # Convert Time column to timedelta
    #df['Time'] = pd.to_timedelta(df['Time'])
    # Rank by Category and Time
    df['Cat Pos'] = df.groupby('Cat')['Time'].rank(method='min', ascending=True).astype(int)

    # Return the dataframe with the new 'Cat Pos' column
    return df

# Loop through each dataframe in the dictionary and apply the 'Cat Pos' function
def get_carnethies(df):
    df = df[df["Club"].str.startswith("Carnethy")]
    return df
    
for race, df in data.items():
    if race in NON_SHR:
        continue
    df = get_carnethies(df)
    data[race] = add_cat_pos(df)

data["Tinto"]

,Pos,Runner,Club,Cat,Time,Cat Pos
0,1,Joseph Wright,Carnethy HRC,M,00:30:44,1
2,3,Alistair Masson,Carnethy HRC,M,00:31:58,2
4,5,Aidan Smith,Carnethy HRC,M,00:32:51,3
8,9,Eliot Sedman,Carnethy HRC,M40,00:35:34,1
10,11,Ifan Oldfield,Carnethy HRC,M,00:35:53,4
15,16,Andrew Macrae,Carnethy HRC,M50,00:36:31,1
17,18,Iain Gilmore,Carnethy HRC,M,00:37:06,5
18,19,Andrew Lamont,Carnethy HRC,M40,00:37:14,2
20,21,Samuel Thom,Carnethy HRC,M,00:37:37,6
22,23,Drew Sharkey,Carnethy HRC,M50,00:37:48,2


In [58]:
data

dict_keys(['Cobbler', 'Mamores', 'Traprain', 'Allermuir', 'Arrochar', 'Blisco', 'Moffat', 'Caerketton', 'Tap O North', 'Perris Horseshoe', 'Caerketton Down', 'Run O Mill', 'Manor Water', 'Skyloop', 'Tinto'])

In [63]:

# Example of your data dictionary (replace this with your actual data)
# Assuming data is a dictionary where keys are race names and values are dataframes
# Also assuming that each dataframe already has the 'Cat Pos' column populated
# Example: data = {'race1': df1, 'race2': df2, ... }

# Function to create a table with the best 6 races and total points for each runner and category
def create_runner_points_table(data):
    # Initialize an empty dataframe to accumulate the results
    runner_points = pd.DataFrame(columns=['Runner', 'Cat', 'Cat Pos'])

    # Loop through each race in the data dictionary
    for race, df in data.items():
        if race in NON_SHR:
            continue
        print(race, df.columns)

        # Ensure 'Cat Pos' is numeric (convert if necessary)
        df['Cat Pos'] = pd.to_numeric(df['Cat Pos'], errors='coerce')
        
        # Check if conversion worked
        if df['Cat Pos'].isnull().any():
            print(f"Warning: Some 'Cat Pos' values are invalid in {race}, converting them to NaN")
        print(df['Cat Pos'])
        # Group by 'Runner' and 'Cat' and take 'Cat Pos' for each runner in each category
        race_points = df[['Runner', 'Cat', 'Cat Pos']]

        # Append the points to the accumulated table of runner points
        runner_points = pd.concat([runner_points, race_points], ignore_index=True)

    # For each runner and category, we want to get the best 6 results (lowest Cat Pos)
    runner_points_sorted = runner_points.groupby(['Runner', 'Cat']).apply(
        lambda x: x.nsmallest(6, 'Cat Pos')  # Select the best 6 results (lowest Cat Pos)
    ).reset_index(drop=True)

    # Now sum the Cat Pos for each runner and category (best 6 results)
    runner_points_total = runner_points_sorted.groupby(['Runner', 'Cat'])['Cat Pos'].sum().reset_index()

    # Sort by total points (lowest points = best performance)
    runner_points_total = runner_points_total.sort_values(by='Cat Pos', ascending=True).reset_index(drop=True)

    return runner_points_total

# Example usage (assuming 'data' is your dictionary of dataframes):
runner_points_table = create_runner_points_table(data)

# Display the table
print(runner_points_table)


Cobbler Index(['Pos', 'Runner', 'Club', 'Cat', 'Time', 'CatPos', 'Cat Pos'], dtype='object')
4       1
5       2
8       3
9       4
13      1
14      1
15      2
23      1
28      1
29      2
34      3
36      4
37      5
38      6
41      7
45      8
51      9
56     10
58     11
64     12
95     13
106    14
114    15
115     1
117     2
121     3
125     4
151     5
161     1
165     2
174     1
183     2
190     1
Name: Cat Pos, dtype: int64
Mamores Index(['Pos', 'Runner', 'Club', 'Cat', 'Time', 'Cat Pos'], dtype='object')
2      1
11     2
12     3
15     4
18     1
20     1
27     5
38     1
68     2
74     1
77     3
101    2
120    2
Name: Cat Pos, dtype: int64
Traprain Index(['Pos', 'Runner', 'Club', 'Cat', 'Time', 'Cat Pos'], dtype='object')
0     1
6     1
8     2
10    1
13    3
14    2
19    2
20    4
24    5
28    6
31    3
35    4
37    1
38    1
40    2
42    1
53    3
56    3
61    1
68    1
69    2
72    7
Name: Cat Pos, dtype: int64
Allermuir Index(['Pos', 'Runner',

TypeError: Column 'Cat Pos' has dtype object, cannot use method 'nsmallest' with this dtype